# CoT Generation for Nemotron Reasoning Challenge

Generates Chain-of-Thought reasoning for all 9,500 training samples using Qwen2.5-72B-Instruct-AWQ.

**Runtime requirements:**
- GPU: RTX Pro 6000 (48GB) or equivalent
- Internet: ON (to download model from HuggingFace)
- Expected runtime: ~60-90 min

**Output:** `/kaggle/working/train_with_cot.csv` (id, prompt, answer, type, generated_cot)

## Cell 1: Configuration

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model — fallback order if a model is unavailable on HuggingFace
# Primary: Qwen2.5-72B-Instruct-AWQ (best quality, ~40GB)
# Fallback: Qwen2.5-32B-Instruct-AWQ (~20GB)
MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct-AWQ"

# Generation parameters
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 3072   # CoT body; leave headroom for prompt
MAX_MODEL_LEN = 8192

# vLLM batch size — tune down if OOM
BATCH_SIZE = 16

# Input / output paths
INPUT_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"
OUTPUT_CSV = "/kaggle/working/train_with_cot.csv"

# Random seed for reproducibility
SEED = 42

print(f"Model : {MODEL_NAME}")
print(f"Input : {INPUT_CSV}")
print(f"Output: {OUTPUT_CSV}")

## Cell 2: Install vLLM

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    import vllm
    print(f"vllm already installed: {vllm.__version__}")
except ImportError:
    print("Installing vllm ...")
    pip_install("vllm")
    import vllm
    print(f"vllm installed: {vllm.__version__}")

## Cell 3: Load Model

In [ ]:
import torch
from vllm import LLM, SamplingParams

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

print(f"\nLoading {MODEL_NAME} ...")
llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
    seed=SEED,
)
tokenizer = llm.get_tokenizer()
print("Model loaded.")

## Cell 4: Load Data & Classify Types

In [ ]:
import csv
from collections import Counter

def classify_type(prompt: str) -> str:
    """Infer puzzle type from prompt text."""
    p = prompt.lower()
    if "bit manipulation" in p:
        return "Bit Manipulation"
    if "encryption" in p or "secret message" in p:
        return "Text Encryption"
    if "transformation rules" in p:
        return "Equation Transformation"
    if "gravitational" in p or "gravity" in p:
        return "Gravitational Constant"
    if "unit" in p and "conversion" in p:
        return "Unit Conversion"
    if "numeral" in p or "roman" in p or "number system" in p:
        return "Numeral Conversion"
    return "Unknown"


rows = []
with open(INPUT_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        row["type"] = classify_type(row["prompt"])
        rows.append(row)

type_counts = Counter(r["type"] for r in rows)
print(f"Total rows: {len(rows)}")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

unknown = [r for r in rows if r["type"] == "Unknown"]
if unknown:
    print(f"\nWARNING: {len(unknown)} rows could not be classified!")
    for r in unknown[:3]:
        print(" ", r["prompt"][:120])
else:
    print("\nAll rows classified successfully.")

## Cell 5: Answer Checker Functions

In [ ]:
import re
from typing import Optional


def extract_boxed(text: str) -> Optional[str]:
    """Extract the last \\boxed{...} value from generated text."""
    # Match \boxed{...} — handle nested braces up to depth 2
    pattern = r'\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}'
    matches = re.findall(pattern, text)
    return matches[-1].strip() if matches else None


def is_correct(pred: Optional[str], truth: str, puzzle_type: str) -> bool:
    """Check if prediction matches ground truth for the given puzzle type."""
    if pred is None:
        return False
    pred = pred.strip()
    truth = truth.strip()

    if puzzle_type == "Bit Manipulation":
        # Exact 8-bit binary match
        return pred == truth

    if puzzle_type in ("Gravitational Constant", "Unit Conversion"):
        # Numeric match: absolute ≤0.05 OR relative ≤0.5%
        try:
            p = float(pred.replace(",", "."))
            t = float(truth.replace(",", "."))
            if abs(t) < 1e-9:
                return abs(p - t) < 1e-9
            return abs(p - t) <= 0.05 or abs(p - t) / abs(t) <= 0.005
        except ValueError:
            return pred.lower() == truth.lower()

    if puzzle_type == "Numeral Conversion":
        return pred.upper() == truth.upper()

    if puzzle_type == "Text Encryption":
        return pred.lower() == truth.lower()

    if puzzle_type == "Equation Transformation":
        return pred == truth

    # Fallback: case-insensitive exact match
    return pred.lower() == truth.lower()

## Cell 6: Type-Specific Prompt Builders

In [ ]:
SYSTEM_BASE = (
    "You are a careful, logical problem-solver. "
    "Work through each step explicitly, showing all reasoning. "
    "Put your final answer inside \\boxed{}. "
    "For example: \\boxed{your answer}"
)


SYSTEM_BIT = (
    "You are solving a bit manipulation puzzle. "
    "Approach: analyze each of the 8 output bit positions independently. "
    "For each bit position i (0=leftmost), examine what value appears in all input→output examples "
    "and find the simplest boolean function of the input bits that produces it. "
    "Candidate functions: identity (copy), NOT, AND, OR, XOR with one or two other bits. "
    "Mark a bit CERTAIN if one function explains all examples, AMBIGUOUS otherwise. "
    "Assemble the 8 output bits, then give your final answer inside \\boxed{}. "
    "The answer must be exactly 8 binary digits, e.g. \\boxed{10110011}"
)

SYSTEM_ENCRYPT = (
    "You are solving a text decryption puzzle. "
    "Approach: (1) Build a character-level substitution map from the examples "
    "(encrypted char → plain char). "
    "(2) Apply the map to decrypt each word in the query, character by character. "
    "(3) If a character is not in the map, reason from context. "
    "Show the map you derived and each word's decryption step. "
    "Give your final answer inside \\boxed{}. "
    "The answer is the decrypted phrase (lowercase English words separated by spaces)."
)

SYSTEM_EQUATION = (
    "You are solving an equation transformation puzzle. "
    "Approach: "
    "(1) Look at the operator symbol(s) between the two operands in each example. "
    "(2) Hypothesize what operation the symbol represents. "
    "    Numeric operators to try: concatenation, addition (+), subtraction (-), "
    "    multiplication (×), division (/), modulo (%), digit-wise operations. "
    "    Non-numeric: character-level substitution cipher. "
    "(3) Verify your hypothesis against EVERY example before accepting it. "
    "(4) Apply the confirmed rule to the query. "
    "Show all hypothesis-testing steps. "
    "Give your final answer inside \\boxed{}."
)

SYSTEM_GRAVITY = (
    "You are solving a gravitational constant puzzle. "
    "The formula is d = 0.5 * g * t^2. "
    "Approach: (1) For each example, solve for g = 2*d / t^2. "
    "(2) Average the g values across all examples. "
    "(3) Apply the average g to compute d for the query t. "
    "(4) Round to 2 decimal places. "
    "Show all arithmetic. "
    "Give your final answer inside \\boxed{} as a number rounded to 2 decimal places."
)

SYSTEM_UNIT = (
    "You are solving a unit conversion puzzle. "
    "Approach: (1) Compute the ratio output/input for each example. "
    "(2) Check if the ratio is constant (multiplicative relationship). "
    "(3) If not constant, try an additive offset (output = a*input + b). "
    "(4) Once the relationship is confirmed, apply it to the query value. "
    "(5) Round to 2 decimal places. "
    "Show all calculations. "
    "Give your final answer inside \\boxed{} as a number rounded to 2 decimal places."
)

SYSTEM_NUMERAL = (
    "You are solving a numeral system conversion puzzle. "
    "Approach: (1) Study the input→output examples to identify the conversion rule "
    "(e.g., decimal to Roman numerals, or a custom base conversion). "
    "(2) Verify the rule against all examples. "
    "(3) Apply the rule to convert the query number. "
    "Show your reasoning step by step. "
    "Give your final answer inside \\boxed{}."
)

TYPE_TO_SYSTEM = {
    "Bit Manipulation":       SYSTEM_BIT,
    "Text Encryption":        SYSTEM_ENCRYPT,
    "Equation Transformation": SYSTEM_EQUATION,
    "Gravitational Constant":  SYSTEM_GRAVITY,
    "Unit Conversion":         SYSTEM_UNIT,
    "Numeral Conversion":      SYSTEM_NUMERAL,
    "Unknown":                 SYSTEM_BASE,
}

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)


def build_chat_prompt(row: dict) -> str:
    """Build a tokenizer-formatted prompt string for vLLM."""
    system_msg = TYPE_TO_SYSTEM.get(row["type"], SYSTEM_BASE)
    user_msg = row["prompt"] + PROMPT_SUFFIX
    messages = [
        {"role": "system",  "content": system_msg},
        {"role": "user",    "content": user_msg},
    ]
    # apply_chat_template returns a string
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

print("Prompt builders defined.")
# Quick sanity check on one example
sample = rows[0]
prompt_str = build_chat_prompt(sample)
print(f"Sample type: {sample['type']}")
print(f"Prompt length: {len(prompt_str)} chars")

## Cell 7: Batch CoT Generation

In [ ]:
import time

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)

# Build all prompts
print("Building prompts...")
prompts = [build_chat_prompt(r) for r in rows]
print(f"  Total prompts: {len(prompts)}")

# Run generation in batches
print(f"\nGenerating CoT (batch_size={BATCH_SIZE}) ...")
t0 = time.time()

all_outputs = [None] * len(prompts)

for batch_start in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[batch_start : batch_start + BATCH_SIZE]
    batch_outputs = llm.generate(batch_prompts, sampling_params)
    for i, output in enumerate(batch_outputs):
        all_outputs[batch_start + i] = output.outputs[0].text

    if (batch_start // BATCH_SIZE) % 20 == 0:
        elapsed = time.time() - t0
        done = batch_start + len(batch_prompts)
        pct = done / len(prompts) * 100
        eta = elapsed / max(done, 1) * (len(prompts) - done)
        print(f"  [{done:5d}/{len(prompts)}] {pct:.1f}%  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m")

elapsed_total = time.time() - t0
print(f"\nGeneration complete in {elapsed_total/60:.1f} min.")
print(f"Total output tokens (approx): {sum(len(o.split()) for o in all_outputs):,}")

## Cell 8: Answer Extraction & Correctness Filtering

In [ ]:
from collections import defaultdict

stats = defaultdict(lambda: {"total": 0, "correct": 0, "no_boxed": 0})
verified_rows = []

for row, cot_text in zip(rows, all_outputs):
    t = row["type"]
    stats[t]["total"] += 1

    pred = extract_boxed(cot_text)
    if pred is None:
        stats[t]["no_boxed"] += 1
        continue

    if is_correct(pred, row["answer"], t):
        stats[t]["correct"] += 1
        # Strip any \boxed{} occurrences from the CoT body itself
        # (the training format appends \boxed{answer} separately)
        cot_clean = re.sub(r'\\boxed\{[^}]*\}', '', cot_text).rstrip()
        verified_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "answer": row["answer"],
            "type": t,
            "generated_cot": cot_clean,
        })

print("=== Filtering Results ===")
print(f"{'Type':<28} {'Total':>6} {'Correct':>8} {'Pass%':>7} {'No-\\boxed':>10}")
print("-" * 65)
for t in sorted(stats.keys()):
    s = stats[t]
    rate = s["correct"] / s["total"] * 100 if s["total"] else 0
    print(f"{t:<28} {s['total']:>6} {s['correct']:>8} {rate:>6.1f}% {s['no_boxed']:>10}")
print("-" * 65)
total_correct = sum(s["correct"] for s in stats.values())
total_all = sum(s["total"] for s in stats.values())
print(f"{'TOTAL':<28} {total_all:>6} {total_correct:>8} {total_correct/total_all*100:>6.1f}%")
print(f"\nVerified CoT samples: {len(verified_rows)}")

## Cell 9: Save CSV

In [ ]:
import csv
import os

FIELDNAMES = ["id", "prompt", "answer", "type", "generated_cot"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(verified_rows)

size_mb = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f"Saved {len(verified_rows)} rows to {OUTPUT_CSV} ({size_mb:.1f} MB)")

# Per-type breakdown of saved rows
type_saved = Counter(r["type"] for r in verified_rows)
print("\nPer-type breakdown:")
for t in sorted(type_saved.keys()):
    print(f"  {t}: {type_saved[t]}")

## Cell 10: Spot-check a few samples

In [ ]:
import random
random.seed(SEED)

for t in sorted(type_saved.keys()):
    samples = [r for r in verified_rows if r["type"] == t]
    sample = random.choice(samples)
    cot_preview = sample["generated_cot"][:400].replace("\n", " | ")
    print(f"=== {t} ===")
    print(f"  answer : {sample['answer']}")
    print(f"  cot    : {cot_preview}...")
    print()